# Linear Interpolation

## The Problem

Given two known data points $(x_0, y_0)$ and $(x_1, y_1)$, estimate the value of the underlying
function $f$ at some query point $x^* \in [x_0, x_1]$.

We know $f(x_0) = y_0$ and $f(x_1) = y_1$ but nothing else about $f$.
Linear interpolation makes the simplest possible assumption: **$f$ is a straight line between the two points**.

---

## Deriving the Formula

The unique line through $(x_0, y_0)$ and $(x_1, y_1)$ has slope:

$$m = \frac{y_1 - y_0}{x_1 - x_0}$$

Evaluating the point-slope form at $x^*$:

$$p(x^*) = y_0 + \frac{y_1 - y_0}{x_1 - x_0}(x^* - x_0)$$

Define the **parameter** $t \in [0, 1]$ as the fractional position of $x^*$ within the interval:

$$t = \frac{x^* - x_0}{x_1 - x_0}$$

Then the formula takes its most symmetric form:

$$\boxed{p(x^*) = (1 - t)\,y_0 + t\,y_1}$$

At $t = 0$ we recover $y_0$; at $t = 1$ we recover $y_1$; in between we get a weighted blend.
The weights $(1-t)$ and $t$ are the **Lagrange basis polynomials** of degree 1:

$$\ell_0(x) = \frac{x - x_1}{x_0 - x_1} = 1 - t, \qquad \ell_1(x) = \frac{x - x_0}{x_1 - x_0} = t$$

so that $p(x) = y_0\,\ell_0(x) + y_1\,\ell_1(x)$ — the general Lagrange interpolation formula at degree 1.

---

## The Lagrange Form Makes the Structure Explicit

Each basis function $\ell_i$ equals 1 at its own node and 0 at the other:

$$\ell_0(x_0) = 1,\quad \ell_0(x_1) = 0 \qquad \ell_1(x_0) = 0,\quad \ell_1(x_1) = 1$$

So the interpolant automatically satisfies the interpolation conditions $p(x_i) = y_i$ regardless of the $y$ values — the geometry is entirely encoded in the basis, and the $y$ values just scale it.

---

## Error Analysis

Suppose $f$ is twice differentiable on $[x_0, x_1]$. Define $h = x_1 - x_0$. The interpolation error at any $x^* \in [x_0, x_1]$ is:

$$e(x^*) = f(x^*) - p(x^*) = \frac{f''(\xi)}{2}(x^* - x_0)(x^* - x_1), \quad \xi \in (x_0, x_1)$$

**Proof.** Define the error function $e(x) = f(x) - p(x)$. It vanishes at both endpoints:

$$e(x_0) = 0, \qquad e(x_1) = 0$$

Construct the auxiliary function:

$$\phi(x) = e(x) - \frac{(x - x_0)(x - x_1)}{(x^* - x_0)(x^* - x_1)}\,e(x^*)$$

Then $\phi$ vanishes at $x_0$, $x_1$, and $x^*$ — three distinct points. By Rolle's theorem, $\phi'$ vanishes at (at least) two points in $(x_0, x_1)$, and $\phi''$ vanishes at (at least) one point $\xi \in (x_0, x_1)$. Since $p''(x) = 0$ (the interpolant is linear) and $[(x-x_0)(x-x_1)]'' = 2$:

$$\phi''(\xi) = f''(\xi) - \frac{2\,e(x^*)}{(x^* - x_0)(x^* - x_1)} = 0$$

Solving for $e(x^*)$:

$$\boxed{e(x^*) = \frac{f''(\xi)}{2}(x^* - x_0)(x^* - x_1), \qquad \xi \in (x_0, x_1)}$$

---

## Bounding the Error

The product $(x^* - x_0)(x^* - x_1)$ is maximised in magnitude at the midpoint $x^* = x_0 + h/2$, where it equals $-h^2/4$. So:

$$|e(x^*)| \leq \frac{\|f''\|_\infty}{2} \cdot \frac{h^2}{4} = \frac{h^2}{8}\|f''\|_\infty$$

The error is $O(h^2)$: **halving the interval width quarters the error**.

---

## Piecewise Linear Interpolation

For a table of $n+1$ data points $x_0 < x_1 < \cdots < x_n$, apply linear interpolation on each
subinterval $[x_{i-1}, x_i]$ independently. The resulting **piecewise linear interpolant** is:

$$p(x) = y_{i-1}\,\frac{x_i - x}{h_i} + y_i\,\frac{x - x_{i-1}}{h_i}, \qquad x \in [x_{i-1}, x_i]$$

where $h_i = x_i - x_{i-1}$. With uniform spacing $h$, the global error is:

$$\|f - p\|_\infty \leq \frac{h^2}{8}\|f''\|_\infty$$

This is $O(h^2)$ globally and is the foundation for finite element methods, lookup tables, and signal reconstruction.

---

## Geometric Interpretation: The Parameter $t$

The parameter form $p = (1-t)\,y_0 + t\,y_1$ is a **convex combination** — a weighted average that stays between $y_0$ and $y_1$ for $t \in [0,1]$. This generalises naturally:

- In **2D**: $(1-t)\,\mathbf{P}_0 + t\,\mathbf{P}_1$ traces the line segment between two points — the basis of Bézier curves.
- In **colour blending**: mixing two RGB values by parameter $t$.
- In **animation**: lerping between keyframes.
- In **finite elements**: the hat basis functions are exactly the piecewise linear Lagrange basis.

---

## Comparison with Higher-Order Interpolation

| Method | Polynomial degree | Error order | Notes |
|---|---|---|---|
| Linear | 1 | $O(h^2)$ | Always stable, no oscillation |
| Quadratic | 2 | $O(h^3)$ | Needs 3 points per panel |
| Cubic spline | 3 (piecewise) | $O(h^4)$ | Smooth derivative at joins |
| Degree-$n$ global | $n$ | $O(h^{n+1})$ | Runge's phenomenon for large $n$ |

Linear interpolation is the most robust: it never overshoots, always stays between the data values (monotone data $\Rightarrow$ monotone interpolant), and requires only two points. Higher-order methods buy accuracy at the cost of potential oscillation.

---

## Conditions and Caveats

| Concern | Detail |
|---|---|
| Requires $f \in C^2[x_0, x_1]$ | Error formula needs $f''$ to exist |
| Extrapolation ($x^* \notin [x_0,x_1]$) | Error grows unboundedly; avoid if possible |
| Non-smooth $f$ | If $f$ has a kink, $f'' $ is undefined there — piecewise linear still converges but error formula changes |
| Monotonicity | Piecewise linear always preserves monotonicity; higher-order methods may not |

In [1]:
def linear_interpolation(x0, y0, x1, y1, x):
    """Perform linear interpolation to find the value of y at a given x.

    Args:
        x0 (float): The first x-coordinate.
        y0 (float): The corresponding y-coordinate for x0.
        x1 (float): The second x-coordinate.
        y1 (float): The corresponding y-coordinate for x1.
        x (float): The x-coordinate at which to interpolate.

    Returns:
        float: The interpolated y-coordinate corresponding to the given x.
    """
    if x1 == x0:
        raise ValueError("x0 and x1 cannot be the same value.")
    
    # Calculate the slope (m) of the line connecting the two points
    m = (y1 - y0) / (x1 - x0)
    
    # Use the point-slope form of the line to find the interpolated value
    y = y0 + m * (x - x0)
    
    return y

# Cubic Spline Interpolation

## The Problem

Given $n+1$ data points $(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n)$ with $x_0 < x_1 < \cdots < x_n$,
find a smooth curve passing through all of them.

A single high-degree polynomial through all $n+1$ points suffers from **Runge's phenomenon** —
wild oscillations between the data points, especially near the endpoints. The fix is to use
**low-degree polynomials on each subinterval**, stitched together smoothly at the **knots** $x_i$.
Cubic splines use degree-3 polynomials and enforce continuity of the function, its first derivative,
and its second derivative at every interior knot.

---

## Setup

Define $n$ subintervals $[x_{i-1}, x_i]$ with spacing $h_i = x_i - x_{i-1}$.
On each subinterval, place a cubic polynomial $S_i(x)$:

$$S_i(x) = a_i + b_i(x - x_{i-1}) + c_i(x - x_{i-1})^2 + d_i(x - x_{i-1})^3, \quad x \in [x_{i-1}, x_i]$$

Each of the $n$ pieces has **4 coefficients**, giving $4n$ unknowns total.

---

## The Conditions

We impose conditions to determine all $4n$ unknowns.

**Interpolation at left knots** ($n$ equations):
$$S_i(x_{i-1}) = y_{i-1}, \quad i = 1, \dots, n$$

This gives immediately $a_i = y_{i-1}$.

**Interpolation at right knots** ($n$ equations):
$$S_i(x_i) = y_i, \quad i = 1, \dots, n$$

**Continuity of first derivative** at interior knots ($n-1$ equations):
$$S_{i-1}'(x_i) = S_i'(x_i), \quad i = 1, \dots, n-1$$

**Continuity of second derivative** at interior knots ($n-1$ equations):
$$S_{i-1}''(x_i) = S_i''(x_i), \quad i = 1, \dots, n-1$$

Total so far: $n + n + (n-1) + (n-1) = 4n - 2$ equations for $4n$ unknowns.
We need **2 more conditions** — these are the **boundary conditions**.

---

## Boundary Conditions

The two free conditions pin the behaviour at the endpoints $x_0$ and $x_n$. Common choices:

**Natural spline** (zero curvature at ends):
$$S_1''(x_0) = 0, \qquad S_n''(x_n) = 0$$

**Clamped spline** (prescribed slopes at ends):
$$S_1'(x_0) = f'(x_0), \qquad S_n'(x_n) = f'(x_n)$$

**Not-a-knot** (cubic continues across first and last interior knot):
$$S_1'''(x_1) = S_2'''(x_1), \qquad S_{n-1}'''(x_{n-1}) = S_n'''(x_{n-1})$$

With any of these, the system is exactly $4n \times 4n$ and has a unique solution.

---

## Reducing to a Tridiagonal System

Introduce the **second derivatives at the knots** as unknowns:

$$M_i = S_i''(x_i), \quad i = 0, 1, \dots, n$$

Since each $S_i$ is cubic, $S_i''$ is linear, so on $[x_{i-1}, x_i]$:

$$S_i''(x) = M_{i-1}\frac{x_i - x}{h_i} + M_i\frac{x - x_{i-1}}{h_i}$$

Integrating twice and applying the interpolation conditions $S_i(x_{i-1}) = y_{i-1}$, $S_i(x_i) = y_i$:

$$S_i(x) = M_{i-1}\frac{(x_i - x)^3}{6h_i} + M_i\frac{(x - x_{i-1})^3}{6h_i} + \left(y_{i-1} - \frac{M_{i-1}h_i^2}{6}\right)\frac{x_i - x}{h_i} + \left(y_i - \frac{M_i h_i^2}{6}\right)\frac{x - x_{i-1}}{h_i}$$

Now enforce $S_{i-1}'(x_i) = S_i'(x_i)$ at each interior knot. After simplification, this yields one equation per interior knot $i = 1, \dots, n-1$:

$$\boxed{h_i M_{i-1} + 2(h_i + h_{i+1})M_i + h_{i+1}M_{i+1} = 6\left(\frac{y_{i+1} - y_i}{h_{i+1}} - \frac{y_i - y_{i-1}}{h_i}\right)}$$

This is a **tridiagonal linear system** in the $M_i$:

$$\begin{bmatrix}
2(h_1+h_2) & h_2 & & \\
h_2 & 2(h_2+h_3) & h_3 & \\
& \ddots & \ddots & \ddots \\
& & h_{n-1} & 2(h_{n-1}+h_n)
\end{bmatrix}
\begin{bmatrix} M_1 \\ M_2 \\ \vdots \\ M_{n-1} \end{bmatrix}
=
\begin{bmatrix} r_1 \\ r_2 \\ \vdots \\ r_{n-1} \end{bmatrix}$$

where $r_i = 6\!\left(\dfrac{y_{i+1}-y_i}{h_{i+1}} - \dfrac{y_i - y_{i-1}}{h_i}\right)$ and the natural boundary conditions supply $M_0 = M_n = 0$.

The matrix is **strictly diagonally dominant**, so the system is guaranteed to have a unique solution and can be solved in $O(n)$ operations by the Thomas algorithm.

---

## Recovering the Coefficients

Once the $M_i$ are known, the coefficients of each piece $S_i$ follow directly:

$$a_i = y_{i-1}$$

$$b_i = \frac{y_i - y_{i-1}}{h_i} - \frac{h_i}{6}(2M_{i-1} + M_i)$$

$$c_i = \frac{M_{i-1}}{2}$$

$$d_i = \frac{M_i - M_{i-1}}{6h_i}$$

---

## Error Analysis

For a natural or clamped cubic spline interpolating $f \in C^4[a,b]$ at equally-spaced knots with spacing $h$:

$$\|f - S\|_\infty \leq \frac{5}{384} h^4 \|f^{(4)}\|_\infty$$

The error is $O(h^4)$: **halving the knot spacing reduces the error by a factor of 16**. This matches the order of Simpson's rule and is achieved without a global high-degree polynomial.

Derivatives are also well approximated:

$$\|f' - S'\|_\infty \leq \frac{h^3}{24}\|f^{(4)}\|_\infty, \qquad \|f'' - S''\|_\infty \leq \frac{3h^2}{8}\|f^{(4)}\|_\infty$$

---

## Why Cubic? The Minimum Curvature Property

Among all functions $g \in C^2[a,b]$ that interpolate the data, the natural cubic spline **minimises total curvature**:

$$\int_a^b \bigl[g''(x)\bigr]^2\,dx \geq \int_a^b \bigl[S''(x)\bigr]^2\,dx$$

This is why splines were originally used in drafting: a physical spline (a flexible strip of wood or metal forced through fixed points) naturally minimises bending energy $\propto \int (g'')^2$, and the natural cubic spline is its exact mathematical counterpart.

---

## Counting Degrees of Freedom

| Source | Count |
|---|---|
| Total unknowns ($4$ per piece, $n$ pieces) | $4n$ |
| Interpolation at all knots | $2n$ |
| $C^1$ at interior knots | $n-1$ |
| $C^2$ at interior knots | $n-1$ |
| Boundary conditions | $2$ |
| **Total equations** | $4n$ |

The system is exactly determined — no freedom remains after the boundary conditions are imposed.

---

## Comparison with Other Interpolants

| Method | Smoothness | Error order | Risk of oscillation |
|---|---|---|---|
| Piecewise linear | $C^0$ | $O(h^2)$ | None |
| Piecewise quadratic | $C^1$ | $O(h^3)$ | Low |
| **Natural cubic spline** | $C^2$ | $O(h^4)$ | Very low |
| Global degree-$n$ poly | $C^\infty$ | $O(h^{n+1})$ | High (Runge) |
| Cubic Hermite (PCHIP) | $C^1$ | $O(h^4)$ | None (monotone-preserving) |

---

## Conditions and Caveats

| Concern | Detail |
|---|---|
| Boundary condition choice matters | Natural spline can droop at endpoints if data has curvature there; clamped is more accurate if $f'$ is known |
| Not monotonicity-preserving | Spline may overshoot between data points; use PCHIP if monotonicity is required |
| Non-uniform knots | The tridiagonal structure still holds; just use variable $h_i$ |
| Extrapolation | Cubic pieces can diverge rapidly outside $[x_0, x_n]$; clamp or use linear extension |
| Higher dimensions | Tensor-product splines extend to 2D surfaces; used in CAD, computer graphics, and finite elements |

In [2]:
def cubic_spline_interpolation_naive(x0, y0, x1, y1, x):
    """Perform cubic spline interpolation to find the value of y at a given x.

    Args:
        x0 (float): The first x-coordinate.
        y0 (float): The corresponding y-coordinate for x0.
        x1 (float): The second x-coordinate.
        y1 (float): The corresponding y-coordinate for x1.
        x (float): The x-coordinate at which to interpolate.

    Returns:
        float: The interpolated y-coordinate corresponding to the given x.
    """
    # For simplicity, we will assume that the cubic spline is defined by the two points
    # and that the second derivatives at these points are zero (natural spline).
    
    # Calculate the coefficients of the cubic polynomial
    a = y0
    b = 0  # Since the second derivative is zero at both points
    c = 3 * (y1 - y0) / ((x1 - x0) ** 2)
    d = -2 * (y1 - y0) / ((x1 - x0) ** 3)
    
    # Calculate the interpolated value using the cubic polynomial
    dx = x - x0
    y = a + b * dx + c * (dx ** 2) + d * (dx ** 3)
    
    return y